###General Linear Regression and Statistical Inference

In [5]:
from google.colab import files
uploaded = files.upload()

Saving cal_housing.tgz to cal_housing.tgz


In [30]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import os
import tarfile
import joblib # Import joblib directly
from sklearn.datasets._base import _pkl_filepath, get_data_home
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns

archive_path = "cal_housing.tgz" # change the path if it's not in the current directory
data_home = get_data_home(data_home=None) # change data_home if you are not using ~/scikit_learn_data
if not os.path.exists(data_home):
    os.makedirs(data_home)
filepath = _pkl_filepath(data_home, 'cal_housing.pkz')

with tarfile.open(mode="r:gz", name=archive_path) as f:
    cal_housing = np.loadtxt(
        f.extractfile('CaliforniaHousing/cal_housing.data'),
        delimiter=',')
    # Columns are not in the same order compared to the previous
    # URL resource on lib.stat.cmu.edu
    columns_index = [8, 7, 2, 3, 4, 5, 6, 1, 0]
    cal_housing = cal_housing[:, columns_index]

    joblib.dump(cal_housing, filepath, compress=6) # Now using the directly imported joblib


# Load the dataset
california = fetch_california_housing(as_frame=True)
data = california.data
data['MedianHouseValue'] = california.target

Print the basic information of the data using `.info()` and `.describe`.

In [31]:
# Display basic information

data.info()
data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   MedInc            20640 non-null  float64
 1   HouseAge          20640 non-null  float64
 2   AveRooms          20640 non-null  float64
 3   AveBedrms         20640 non-null  float64
 4   Population        20640 non-null  float64
 5   AveOccup          20640 non-null  float64
 6   Latitude          20640 non-null  float64
 7   Longitude         20640 non-null  float64
 8   MedianHouseValue  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedianHouseValue
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


### Step 1

Let `X` be the variables `MedInc`, `AveRooms`, and `AveOccup` and add the constant for the intercept. Let `y` be the `MedianHouseValue`.

Now fit the regreson model calling it `mlr_model`.

Finally, return the $r^2$ value of the model rounding to four decimal places.

In [32]:
X = data[['MedInc', 'AveRooms', 'AveOccup']]
X = sm.add_constant(X)

y = data['MedianHouseValue']

mlr_model = sm.OLS(y, X).fit()

round(mlr_model.rsquared, 4)


np.float64(0.4808)

Print the model summary.

In [33]:
print(mlr_model.summary())

                            OLS Regression Results                            
Dep. Variable:       MedianHouseValue   R-squared:                       0.481
Model:                            OLS   Adj. R-squared:                  0.481
Method:                 Least Squares   F-statistic:                     6370.
Date:                Sat, 19 Sep 2026   Prob (F-statistic):               0.00
Time:                        06:39:16   Log-Likelihood:                -25477.
No. Observations:               20640   AIC:                         5.096e+04
Df Residuals:                   20636   BIC:                         5.099e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6069      0.016     37.444      0.0

### Step 2

Let `p_values` be the models' p-values.

Return the four p-values using `.iloc[]` from the first value to the fourth, in order and separated by commas. Make sure to round each to 5 decimal places.

In [34]:
p_values = mlr_model.pvalues

round(p_values.iloc[0], 5), round(p_values.iloc[1], 5), round(p_values.iloc[2], 5), round(p_values.iloc[3], 5)

(np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0))

### Step 3

Identify the significant predictors (strictly less than $\alpha=0.05$) calling this `significant_predictors`.

Reutn the shape of `significant_predictors`.

In [35]:
significant_predictors = mlr_model.pvalues[mlr_model.pvalues < 0.05]

significant_predictors.shape

(4,)

### Step 4

Find the confidence intervals of the model (at a 95% level of confidence) and calling this `conf_intervals`.

Using `.iloc[,]` and rounding to 2 decimal places return the four confidence intervals in order of (separated by commas)

> first row and first column, first row and second column, second row and first column, second row and second column





In [36]:
conf_intervals = mlr_model.conf_int(alpha=0.05)

round(conf_intervals.iloc[0, 0], 2), round(conf_intervals.iloc[0, 1], 2), round(conf_intervals.iloc[1, 0], 2), round(conf_intervals.iloc[1, 1], 2)

(np.float64(0.58), np.float64(0.64), np.float64(0.43), np.float64(0.44))

Now to see how the intervals looks "nicely" return `conf_intervals`.

In [37]:
conf_intervals


,0,1
const,0.575162,0.638703
MedInc,0.428363,0.441003
AveRooms,-0.043178,-0.033474
AveOccup,-0.005266,-0.003081


### Step 5

Add a quadratic term to the model, calling the new model `quad_model` where a new term is added to the data, viz. `MedInc_squared`, which is the square of `MedInc`.

Return $r^2$ of the quadratic model rounded to four decumal places.

In [38]:
data['MedInc_squared'] = data['MedInc'] ** 2

X_quad = data[['MedInc', 'AveRooms', 'AveOccup', 'MedInc_squared']]
X_quad = sm.add_constant(X_quad)

quad_model = sm.OLS(y, X_quad).fit()

round(quad_model.rsquared, 4)


np.float64(0.4858)

Now print the model summary.

In [39]:
print(quad_model.summary())


                            OLS Regression Results                            
Dep. Variable:       MedianHouseValue   R-squared:                       0.486
Model:                            OLS   Adj. R-squared:                  0.486
Method:                 Least Squares   F-statistic:                     4874.
Date:                Sat, 19 Sep 2026   Prob (F-statistic):               0.00
Time:                        06:39:26   Log-Likelihood:                -25378.
No. Observations:               20640   AIC:                         5.077e+04
Df Residuals:                   20635   BIC:                         5.081e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.3551      0.024     14.

### Step 6

Find the adjusted $r^2$ for both of the models and call them `adjusted_r2_base` and `adjusted_r2_quad`, respectively.

Return these two adjusted $r^2$'s rounded to four decimal places, separated by a comma.

In [40]:
adjusted_r2_base = mlr_model.rsquared_adj
adjusted_r2_quad = quad_model.rsquared_adj

round(adjusted_r2_base, 4), round(adjusted_r2_quad, 4)

(np.float64(0.4807), np.float64(0.4857))

Print both these adjusted $r^2$'s.

In [41]:
print(adjusted_r2_base)
print(adjusted_r2_quad)

0.48074591990192506
0.4856862487881798
